# N020 · 频率统计与等价类分组

**目标：** 为对象构造不丢失等价信息的键。

**先修：** N006, N007, N019。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 频次数组；排序签名；哈希键；Counter；字符集前提。

**配套讲解来源：** [同名逐章意见](../../comment/020_hash_frequency_grouping.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章围绕"异位词"展开两个问题。第一个：**判断两个词是不是异位词**——两个词用的字符完全一样（每个字符出现次数都相同），只是排列顺序不同。第二个：**把一堆词按异位词关系分组**——所有互为异位词的词放进同一组，扫完整个列表输出全部组。

举两个具体到数字的例子：

- 输入 `s='anagram'`、`t='nagaram'` → 输出 `True` → 为什么？逐个数一下，两个词的字符账本完全一样：

| 字符 | a | g | m | n | r |
|---|---|---|---|---|---|
| anagram 中次数 | 3 | 1 | 1 | 1 | 1 |
| nagaram 中次数 | 3 | 1 | 1 | 1 | 1 |

每个字符出现次数完全相同，只是位置被打乱了，所以它们是异位词。

- 输入 `words=['eat','tea','tan','ate','nat','bat']` → 输出三组 `[['eat','tea','ate'], ['tan','nat'], ['bat']]` → 为什么？eat、tea、ate 都由 e、a、t 各一个组成，归为一组；tan、nat 都由 t、a、n 各一个组成，归为一组；bat 没有同伴，自己单独一组。

注意一个容易被忽略的细节：分组时如果列表里出现重复单词（比如两个 'eat'），它们也要各自入组，不能像集合那样合并掉一个。

## 2. 基础知识：先读懂这些词

- **异位词（anagram）**：字符多重集合相同的两个字符串。"多重集合"听着玄，意思就是"有哪些字符、每个出现几次"完全一样，但不管顺序。
- **频次 / 频率（frequency）**：某个字符出现的次数。异位词判断的本质就是"两边每个字符的频次逐一相等"。
- **Counter**：Python 标准库 `collections` 里的计数器，你把一个字符串丢给它，它就返回一个"字符 → 出现次数"的字典。比如 `Counter('eat')` 得到 `{'e':1,'a':1,'t':1}`。
- **排序签名（sorted signature）**：把单词的字符按字典序排好、拼成一个标准形式。'eat'、'tea'、'ate' 排完都是 'aet'。这个标准形式就是单词的"签名"——互为异位词的词签名必然相同，不是异位词签名必然不同。
- **哈希键（hash key）**：字典依据它来分组。签名可以直接当字典的键使用，字典自动把同键的值归拢到一起，分组就白送了。
- **defaultdict(list)**：一种"缺席也不报错"的字典——访问一个还不存在的键时，它自动先给你造一个空列表。用它做分组收集器可以省掉"先判断键在不在"的样板代码。
- **频次数组（26 维计数）**：如果题目保证只有小写字母，也可以用长度 26 的数组代替字典，下标 0~25 对应 a~z，格子里存出现次数。这是"字符集前提"带来的优化空间。
- **等价类 / 等价关系**：如果"谁和谁算同类"这个关系满足自反、对称、传递（每个词和自己同类；A 与 B 同类则 B 与 A 同类；A 与 B 同类、B 与 C 同类则 A 与 C 同类），它就把全体单词切成若干个互不重叠的组，每组叫一个等价类。"签名相同"正是这样一套关系，所以按签名分出来的组是干净利落的。

## 3. 思路推导：从小例子开始

- **Step 1：抓住不变量。** 异位词忽略位置，但保留"每个字符出现几次"。所以判断异位词不应该比较排列，而应该比较频次。
- **Step 2：把词换成规范表示。** 要么数频次（Counter），要么把字符排序。两种"规范表示"都能抹掉位置信息、只留等价信息：互为异位词 → 规范表示相同；不是 → 必不相同。
- **Step 3：判断题直接比规范表示。** `is_anagram` 就是"两个 Counter 是否相等"一句话。
- **Step 4：分组题用规范表示当字典键。** 扫描词表，每个词算出排序签名，往 `groups[签名]` 里追加原词。字典天然把同签名的词攒到一起。
- **Step 5：追加原词而不是丢进集合。** 用 `list.append` 收集，重复词就都留得下来；分组顺序、组内顺序取决于扫描顺序，但那不是问题的本质，比较前先规范化即可。

**手算演示**（对应 “运行示例”部分 的表格，`words=['eat','tea','tan','ate','nat','bat']`）：先给每个词算排序签名。

| 单词 | 排序过程 | 规范键 |
|---|---|---|
| eat | e,a,t → 排序 | aet |
| tea | t,e,a → 排序 | aet |
| tan | t,a,n → 排序 | ant |
| ate | a,t,e → 排序 | aet |
| nat | n,a,t → 排序 | ant |
| bat | b,a,t → 排序 | abt |

然后按键入账：键 'aet' 收到 [eat, tea, ate]，键 'ant' 收到 [tan, nat]，键 'abt' 收到 [bat]。输出这三组，与 “运行示例”部分 第二张表格逐项一致。

顺带看一眼"排序签名"和"频次签名"在同一个词上的长相（后者是练习 1 要用的）：

| 词 | 排序签名 | 频次签名（仅列出现的字符:次数） |
|---|---|---|
| eat | `('a','e','t')` | `{a:1, e:1, t:1}` |
| tea | `('a','e','t')` | `{a:1, e:1, t:1}` |
| tan | `('a','n','t')` | `{a:1, n:1, t:1}` |

两种签名都做到了"位置信息被抹掉、次数信息被保留"，所以 eat 和 tea 签名相同、eat 和 tan 签名不同——目标一致，只是记账格式不同。

## 4. 核心代码：is_anagram

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def is_anagram(s, t):
    return Counter(s) == Counter(t)
```

### 逐段读懂代码

### 准备工作

```python
from collections import Counter, defaultdict
```

这行从标准库引入两个工具：`Counter`（自动计数器）和 `defaultdict`（带默认值的字典），下面两个函数各用一个。

### 函数 1：`is_anagram`（判断两个字符串是否互为异位词）

```python
def is_anagram(s, t):
    return Counter(s) == Counter(t)
```

- 整个函数就一句话。`Counter(s)` 把 s 变成"字符 → 次数"的计数表，`Counter(t)` 同理；两个计数表相等，当且仅当每个字符在两边的出现次数都一样——这正是异位词的定义，所以直接返回比较结果即可。
- 顺带注意它对空字符串天然成立：`Counter('')==Counter('')==Counter('')`，两个空串会被判为异位词（本章测试里有这条）。
- 这个实现不挑字符集：小写字母、中文、emoji 都能数，因为它没有预设字符表（对比之下"26 维频次数组"写法只适用于纯小写字母）。

### 函数 2：`group_anagrams`（按异位词关系分组）

先看扫描与收集的主体：

```python
def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        groups[tuple(sorted(word))].append(word)
```

- `groups=defaultdict(list)`：造一个"缺键自动补空列表"的字典，作为分组收集器。第一次访问某个新签名时，defaultdict 先自动造出空列表，所以可以直接 `.append`，不用先写"键在不在"的判断。
- `tuple(sorted(word))` 是构造键的关键一步：`sorted(word)` 把字符排好序、得到一个字符列表，比如 `sorted('eat')` 得 `['a','e','t']`；外层再套 `tuple(...)` 把列表转成元组。为什么要转？因为 Python 的字典键必须不可变，列表不能当键，元组可以。这一步保证键既携带了签名信息、又能合法地做键。
- `.append(word)`：注意追加的是**原词**而不是签名——组里要留的是原始输入；append 也不会去掉重复，重复词全部保留。

再看返回：

```python
return list(groups.values())
```

- `list(groups.values())`：字典的 values() 是各个组（列表），套一层 list 变成标准的"组的列表"返回。组的先后顺序由插入顺序决定，但如推导所说，分组题的本质是"谁和谁同组"，顺序无意义，测试时规范化即可。

## 5. 分段实现：按顺序运行

**本章接口：** `is_anagram(s, t)`、`group_anagrams(words)`

### 导入本章使用的库

In [1]:
from collections import Counter, defaultdict

### is_anagram

In [2]:
def is_anagram(s, t):
    return Counter(s) == Counter(t)

### group_anagrams

In [3]:
def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        groups[tuple(sorted(word))].append(word)
    return list(groups.values())

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

words=['eat','tea','tan','ate','nat','bat']
show_table(['单词','规范键'],[(w,''.join(sorted(w))) for w in words])
show_table(['组号','组'],list(enumerate(group_anagrams(words))))

单词,规范键
eat,aet
tea,aet
tan,ant
ate,aet
nat,ant
bat,abt


组号,组
0,"['eat', 'tea', 'ate']"
1,"['tan', 'nat']"
2,['bat']


## 7. 正确性、适用条件与复杂度

两个词的排序字符序列相同，当且仅当所有字符的出现次数相同。这构成等价关系；每次追加原词而非放入集合，保留重复词。

**代价：** 比较平均 O(|s|+|t|)；分组 O(Σ Lᵢ log Lᵢ) 时间，键与输出存储 O(ΣLᵢ)。

### 展开理解

**为什么是对的：** 先看签名为什么可靠。一个词的字符排完序之后，每个字符都按固定次序排好了，两个词的排序结果完全相同，当且仅当它们拥有的字符种类和每个字符的数量都一模一样——这恰好就是异位词的定义。所以"签名相同"和"互为异位词"是同一件事的两种说法，按签名分组不会错分也不会漏分。再看为什么"不错分"：不同字符组成的词排序后一定在某一位上出现不同的字符，签名必不相同，自然落进不同的键。为什么"不漏分"：每个词都被扫描到、都算了签名、都被追加进了自己签名对应的组。最后，"签名相同"这套关系满足：每个词和自己签名相同；A 与 B 签名相同则 B 与 A 也相同；A 与 B 相同、B 与 C 相同则 A 与 C 相同——所以分出来的组互不重叠、又覆盖所有词，每类恰好一组。

**复杂度是多少：** `is_anagram` 要把两个串各数一遍，时间约 O(|s|+|t|)（|s|、|t| 是两个串的长度），这是"数一遍每个字符"的必然代价。分组要给每个词排序：长度为 L 的词排序花 O(L log L)，所有词加起来是 O(Σ Lᵢ log Lᵢ)；存储所有键和输出一共 O(Σ Lᵢ)。举个具体的数：1000 个长度 100 的词，排序部分约 1000×100×log100≈七十万次比较量级，一瞬间完成。若字符集固定为 26 个小写字母，可以把排序换成数频次（每词只需 O(L)），整体能再降一档——这正是练习 1 要你对比的两种签名。

**两个高频疑问，顺便说清：**

- 疑问一："直接比较 `sorted(s)==sorted(t)` 判断异位词行不行？"——行，而且很容易懂；代价是每次比较都要重新排两个串，O(L log L)。数频次（Counter）是 O(L)，串很长时更快。两种都是合法解，知道取舍就行。
- 疑问二："为什么分组要用 defaultdict，普通 dict 不行吗？"——行，只是普通 dict 第一次遇到新键会报 KeyError，你得补一句"如果键不存在就先建空列表"；defaultdict 把这句样板话吸收掉了，逻辑更聚焦。

## 8. 单元测试：每个用例在检查什么

```python
assert is_anagram('anagram', 'nagaram')

assert not is_anagram('rat', 'car')

def normalized(groups):
    return sorted((sorted(g) for g in groups))

assert normalized(group_anagrams(['eat', 'tea', 'eat', 'bat'])) == [['bat'], ['eat', 'eat', 'tea']]

assert group_anagrams([]) == []

assert is_anagram('', '')

print('N020: 所有本章断言通过')
```

- 第 1 条 `assert is_anagram('anagram','nagaram')`：**正常值测试**（也是 LeetCode 242 的官方样例）。两个词字符集合相同、次数相同，应为真。
- 第 2 条 `assert not is_anagram('rat','car')`：**反例测试**。两词长度相同、大部分字符相同，但 r、a、t 与 r、a、c 在 t/c 上不一致——它考察"逐字符计数"而不是"长度相同+有交集"这种粗糙判断。
- `def normalized(groups): return sorted(sorted(g) for g in groups)`：定义一个规范化函数，干两件事：
  - 内层 `sorted(g)` 把每**组内**的词按字典序排好；
  - 外层 `sorted(...)` 再把**组与组**之间按排好的样子排序。
  - 为什么要它？因为 `group_anagrams` 的输出顺序取决于扫描顺序，断言精确顺序就是在测无关紧要的实现细节；规范化之后，"哪些词在同一组"这个本质信息保留，顺序噪声被洗掉。
- 第 3 条是**重复词分组测试**：输入 `['eat','tea','eat','bat']` 有两个 'eat'，期望规范化后为 `[['bat'],['eat','eat','tea']]`。这条专门钉住"追加原词、保留重复"的契约——如果谁用集合去实现分组，两个 eat 只剩一个，这条当场失败。
- 第 4 条 `assert group_anagrams([])==[]`：**空输入测试**。空列表没有任何词，应返回空的结果列表。
- 第 5 条 `assert is_anagram('','')`：**空串边界测试**。两个空串没有任何字符，频次表都是空的、天然相等，约定它们互为异位词。

In [5]:
assert is_anagram('anagram', 'nagaram')

assert not is_anagram('rat', 'car')

def normalized(groups):
    return sorted((sorted(g) for g in groups))

assert normalized(group_anagrams(['eat', 'tea', 'eat', 'bat'])) == [['bat'], ['eat', 'eat', 'tea']]

assert group_anagrams([]) == []

assert is_anagram('', '')

print('N020: 所有本章断言通过')

N020: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较排序签名和计数签名。

**练习 2：** 支持非ASCII字符时调整实现。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（比较排序签名和计数签名）**：思路方向——把 `group_anagrams` 的键从 `tuple(sorted(word))` 换成"频次计数"形式。提示：小写字母前提下可以用长度 26 的计数元组当键（比如 'eat' 对应 (1,0,0,...,1 在 e,...,1 在 t)，具体下标按字母位置），也可继续用 `tuple(sorted(Counter(word).items()))` 这种通用计数键。对比维度：时间上计数 O(L) 优于排序 O(L log L)；空间上计数键固定 26 格、排序键长 L；适用性上计数键的前提是字符集已知且有限。手算例子就用 'eat' vs 'tea'，两种签名都应给出相同结果。
- **练习 2（支持非 ASCII 字符）**：思路方向——想想哪些写法依赖"只有 26 个小写字母"这个前提。提示：26 维频次数组直接给非 ASCII 字符下标会越界或错位，必须换成无固定字符集的方案——`Counter`/字典计数天然支持任意字符，`tuple(sorted(word))` 排序签名也支持（Python 字符串按码点排序）。手算例子可以用中文：'啊好' 与 '好啊' 互为异位词，两种实现都应判 True；最后说明本章原实现（Counter 和排序）本来就不挑字符集，真正需要改造的是那套 26 维数组的假想写法。

两道练习验收时的共同要求：写清"输入、输出、边界"，并且能口头回答"两个不是异位词的词，签名一定不同吗？为什么"——这一句就是签名方案正确性的核心。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter, defaultdict

def is_anagram(s, t):
    return Counter(s) == Counter(t)

def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        groups[tuple(sorted(word))].append(word)
    return list(groups.values())

# 示例与回归测试
assert is_anagram('anagram', 'nagaram')

assert not is_anagram('rat', 'car')

def normalized(groups):
    return sorted((sorted(g) for g in groups))

assert normalized(group_anagrams(['eat', 'tea', 'eat', 'bat'])) == [['bat'], ['eat', 'eat', 'tea']]

assert group_anagrams([]) == []

assert is_anagram('', '')

print('N020: 所有本章断言通过')

N020: 所有本章断言通过


## 11. 代表题与迁移

- **242. Valid Anagram（有效的字母异位词）**：这题就是 `is_anagram` 的原型——练的是"比较频次而不是比较排列"，Counter 和频次数组都是标准解法。
- **49. Group Anagrams（字母异位词分组）**：这题就是 `group_anagrams` 的原型——练的是"构造规范签名当哈希键，把比较问题变成分组问题"，这是哈希思想最经典的应用之一。
- 迁移建议："给对象造一个不丢失等价信息的键"这个思想远不止异位词：按"字母排序"分组、按"数字位计数"找变形词、按"路径规范化"合并重复请求，都是同一个骨架。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。